In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import scipy.io.wavfile as wav
!pip install python_speech_features
from python_speech_features import mfcc

# If your code is giving your problems, remove this
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow.lite")

In [ ]:
# ============================================
# COLAB SETUP
# ============================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully!")
except:
    IN_COLAB = False
    print("Not running in Colab")

In [ ]:
# ============================================
# CONFIGURATION FOR STREAMING DETECTION
# ============================================
SAMPLE_RATE = 16000    # Sample rate of our audio
WINDOW_SIZE_MS = 1250  # 1.25 second window
WINDOW_STRIDE_MS = 20  # 20ms stride (50 inferences/second)
NUM_MFCC = 13          # MFCC parameters
NUM_CLASSES = 3        # Number of categories we train on
CATEGORIES = ['w', 'o', 'n'] # make sure these match your subpaths!

# Paths - you will likely need to update this!
if IN_COLAB:
    DATA_PATH = '/content/drive/MyDrive/EE305/wake_word_dataset/'
    OUTPUT_PATH = '/content/drive/MyDrive/EE305/wake_word_models/'
else:
    DATA_PATH = './data/'
    OUTPUT_PATH = './models/'

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# ============================================
# DATA AUGMENTATION
# ============================================

def add_noise(signal, noise_factor=0.005):
    noise = np.random.randn(len(signal))
    return signal + noise_factor * noise

def time_shift(signal, shift_max=0.5):
    """Aggressive time shift for position invariance"""
    shift = np.random.randint(-int(shift_max * len(signal)),
                               int(shift_max * len(signal)))
    augmented = np.roll(signal, shift)
    if shift > 0:
        augmented[:shift] = 0
    else:
        augmented[shift:] = 0
    return augmented

def change_volume(signal, volume_factor=None):
    if volume_factor is None:
        volume_factor = np.random.uniform(0.5, 1.5)
    return signal * volume_factor

def time_stretch(signal, rate=None):
    if rate is None:
        rate = np.random.uniform(0.8, 1.2)
    indices = np.arange(0, len(signal), rate)
    indices = indices[indices < len(signal)].astype(int)
    stretched = signal[indices]
    if len(stretched) < len(signal):
        stretched = np.pad(stretched, (0, len(signal) - len(stretched)), 'constant')
    else:
        stretched = stretched[:len(signal)]
    return stretched

def add_context_audio(signal, context_type='random'):
    """Add noise before/after the word to *roughly* simulate other conversation"""
    window_length = len(signal)
    threshold = np.max(np.abs(signal)) * 0.1
    voice_indices = np.where(np.abs(signal) > threshold)[0]

    if len(voice_indices) == 0:
        return signal

    start_voice = voice_indices[0]
    end_voice = voice_indices[-1]

    if context_type == 'random' or context_type == 'before':
        if start_voice > 1000:
            noise_length = np.random.randint(500, start_voice)
            noise = np.random.randn(noise_length) * 0.05
            signal[:noise_length] = noise

    if context_type == 'random' or context_type == 'after':
        if end_voice < window_length - 1000:
            noise_start = end_voice + 1
            noise_length = np.random.randint(500, window_length - noise_start)
            noise = np.random.randn(noise_length) * 0.05
            signal[noise_start:noise_start+noise_length] = noise

    return signal

def random_position_in_window(signal):
    """Place word at random position in window (simulates streaming)"""
    threshold = np.max(np.abs(signal)) * 0.1
    voice_indices = np.where(np.abs(signal) > threshold)[0]

    if len(voice_indices) == 0:
        return signal

    start_voice = voice_indices[0]
    end_voice = voice_indices[-1]
    voice_segment = signal[start_voice:end_voice+1]

    window_length = len(signal)
    voice_length = len(voice_segment)

    if voice_length >= window_length:
        return signal

    max_start = window_length - voice_length
    new_start = np.random.randint(0, max_start + 1)

    new_signal = np.zeros(window_length, dtype=signal.dtype)
    new_signal[new_start:new_start+voice_length] = voice_segment

    if np.random.random() > 0.5:
        new_signal = add_context_audio(new_signal,
                                       context_type=np.random.choice(['before', 'after', 'random']))
    return new_signal

def augment_audio(signal):
    """Apply random augmentation"""
    aug_type = np.random.choice(['noise', 'time_shift', 'volume', 'stretch',
                                 'position', 'combined'])

    if aug_type == 'noise':
        return add_noise(signal, np.random.uniform(0.001, 0.01))
    elif aug_type == 'time_shift':
        return time_shift(signal, 0.5)
    elif aug_type == 'volume':
        return change_volume(signal, np.random.uniform(0.6, 1.4))
    elif aug_type == 'stretch':
        return time_stretch(signal, np.random.uniform(0.85, 1.15))
    elif aug_type == 'position':
        return random_position_in_window(signal)
    elif aug_type == 'combined':
        aug_signal = signal.copy()
        for _ in range(np.random.randint(2, 4)):
            aug_signal = augment_audio(aug_signal)
        return aug_signal
    return signal


In [ ]:
# ============================================
# DATA LOADING
# ============================================
def load_dataset(augment=True, aug_factor=5):
    """Load audio files and extract MFCC features"""
    X, y = [], []
    window_samples = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)

    for label_idx, category in enumerate(CATEGORIES):
        path = os.path.join(DATA_PATH, category)
        # This takes a max of 100 files - just the first 100 it finds.
        files = [f for f in os.listdir(path) if f.endswith('.wav')][:100]

        print(f"Loading {category}: {len(files)} files")

        for file in files:
            # Load audio
            rate, signal = wav.read(os.path.join(path, file))
            signal = signal.astype(np.float32) / 32768.0

            # Pad or truncate
            if len(signal) < window_samples:
                signal = np.pad(signal, (0, window_samples - len(signal)))
            else:
                signal = signal[:window_samples]

            # Extract features
            features = mfcc(signal, samplerate=SAMPLE_RATE, numcep=NUM_MFCC)
            X.append(features)
            y.append(label_idx)

            # Augment
            if augment:
                for _ in range(aug_factor - 1):
                    aug_signal = augment_audio(signal)
                    aug_features = mfcc(aug_signal, samplerate=SAMPLE_RATE, numcep=NUM_MFCC)
                    X.append(aug_features)
                    y.append(label_idx)

    return np.array(X)[..., np.newaxis], np.array(y)

In [ ]:
# ============================================
# MODEL
# ============================================
def create_model(input_shape, num_classes):
    """Create lightweight CNN for wake word detection"""
    return keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.SeparableConv2D(32, (3, 3), activation='relu', padding='same'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.2),

        keras.layers.SeparableConv2D(64, (3, 3), activation='relu', padding='same'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.2),

        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(num_classes, activation='softmax')
    ])

In [ ]:
import matplotlib.pyplot as plt

# ============================================
# TRAINING
# ============================================
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=50,
    restore_best_weights=True,
    verbose=1 # Optional: displays a message when stopping
)

def train_model(augment=True, model_name='wake_word_model'):
    """Train wake word model"""
    print("Loading data...")
    X, y = load_dataset(augment, aug_factor=5)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

    print(f"\nTrain: {len(X_train)}, Test: {len(X_test)}")
    print(f"Input shape: {X_train[0].shape}\n")

    model = create_model(X_train[0].shape, len(CATEGORIES))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    print(f"\nTraining model: {model_name} (Augmentation: {augment})...")
    history = model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=32,
              callbacks=[early_stopping])

    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"\nTest accuracy for {model_name}: {test_acc*100:.1f}%")

    model.save(os.path.join(OUTPUT_PATH, f'{model_name}.h5'))
    return model, X_test, y_test, history

In [ ]:
def plot_training_history(history, title):
    """Plot training & validation accuracy and loss"""
    plt.figure(figsize=(12, 5))

    # Plot training & validation accuracy values
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(f'{title} - Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.grid(True)

    # Plot training & validation loss values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(f'{title} - Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================
# TFLITE CONVERSION
# ============================================
def convert_to_tflite(model, X_test, quantize=True, model_name='wake_word'):
    """Convert to TFLite (optionally quantized)"""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantize:
        def representative_dataset():
            for i in range(min(100, len(X_test))):
                yield [X_test[i:i+1].astype(np.float32)]

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    suffix = 'quantized' if quantize else 'float32'
    output_path = os.path.join(OUTPUT_PATH, f'{model_name}_{suffix}.tflite')

    with open(output_path, 'wb') as f:
        f.write(tflite_model)

    print(f"TFLite model saved: {output_path} ({len(tflite_model)/1024:.1f} KB)")
    return output_path

In [ ]:
def test_tflite(tflite_path, X_test, y_test):
    """Test TFLite model accuracy and speed"""
    import time
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_scale, input_zero_point = input_details[0]['quantization']

    correct = 0
    times = []

    for i in range(len(X_test)):
        x = X_test[i:i+1]
        if input_scale > 0:  # Quantized model
            x = x / input_scale + input_zero_point
            x = x.astype(input_details[0]['dtype'])
        else: # float32 model
            x = x.astype(np.float32)

        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], x)
        interpreter.invoke()
        times.append((time.time() - start) * 1000)

        output = interpreter.get_tensor(output_details[0]['index'])
        if np.argmax(output) == y_test[i]:
            correct += 1

    print(f"Accuracy: {correct/len(X_test)*100:.1f}%")
    print(f"Avg inference: {np.mean(times):.2f}ms")

In [ ]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- 1. Baseline Model (no augmentation) ---
print("\n====== Training Baseline Model ======")
baseline_model, X_test_baseline, y_test_baseline, history_baseline = train_model(augment=False, model_name='wake_word_baseline')
plot_training_history(history_baseline, "Baseline Model Training History")

# Convert Baseline Model
print("\n====== Converting Baseline Model (Float32) ======")
baseline_float_path = convert_to_tflite(baseline_model, X_test_baseline, quantize=False, model_name='wake_word_baseline')
print("\n====== Converting Baseline Model (Quantized) ======")
baseline_quantized_path = convert_to_tflite(baseline_model, X_test_baseline, quantize=True, model_name='wake_word_baseline')

# Test Baseline TFLite Models
print("\n====== Testing Baseline Quantized TFLite Model ======")
test_tflite(baseline_quantized_path, X_test_baseline, y_test_baseline)
print("\n====== Testing Baseline Float32 TFLite Model ======")
test_tflite(baseline_float_path, X_test_baseline, y_test_baseline)


# --- 2. Augmented Model (with augmentation) ---
print("\n====== Training Augmented Model ======")
augmented_model, X_test_augmented, y_test_augmented, history_augmented = train_model(augment=True, model_name='wake_word_augmented')
plot_training_history(history_augmented, "Augmented Model Training History")

# Convert Augmented Model
print("\n====== Converting Augmented Model (Float32) ======")
augmented_float_path = convert_to_tflite(augmented_model, X_test_augmented, quantize=False, model_name='wake_word_augmented')
print("\n====== Converting Augmented Model (Quantized) ======")
augmented_quantized_path = convert_to_tflite(augmented_model, X_test_augmented, quantize=True, model_name='wake_word_augmented')

# Test Augmented TFLite Models
print("\n====== Testing Augmented Quantized TFLite Model ======")
test_tflite(augmented_quantized_path, X_test_augmented, y_test_augmented)
print("\n====== Testing Augmented Float32 TFLite Model ======")
test_tflite(augmented_float_path, X_test_augmented, y_test_augmented)